# =========================================
# Day 7 - Practical Integration

- Compare tokenizers
- Analyze token distribution
- Tokenization impact on LLM cost
- Build a preprocessing + tokenization pipeline

# =========================================

---

# =========================================
# 1. INTUITION
# =========================================

Imagine you're running a delivery company:

- Every package = a **token**
- Distance traveled = **cost**
- Route planning = **preprocessing + tokenization pipeline**

If you pack inefficiently (bad tokenization):

- Use more packages (tokens)
- Increase delivery cost (LLM cost)
- Slow down operations (Latency)

#### What problem does this solve?

- Different tokenizers produce **different numbers of tokens**
- Token count directly impacts:
    - Cost (API pricing)
    - Speed (latency)
    - Context limits

#### Why was this needec?
- In real GenAI systems:
    - You pay **per token**
    - You have **context windows limits**
- Efficient proprocessing + tokenization = **cheaper + faster + better systems**

Key Idea: **Better perfomance + lower cost**

---

# =========================================
# 2. CORE CONCEPTS
# =========================================

#### 2.1 Comparing Tokenizers

**Types**

- Word-level (NLTK, spaCy)
- Subword (BPE, WordPiece, SentencePiece)
- LLM-specific tokenizers (tiktoken, HuggingFace)

---

#### **Key Differences**

| **Feature** | **Word Tokenizer** | **Subword Tokenizer** |
|-------------|--------------------|-----------------------|
| OOV | High | Low |
| Token Count | Low | Medium |
| Flexibility | Low | High |

---

#### **2.2 Token Distribution Analysis**

**Definition**

Understanding how tokens are distributed in text:
- Frequeny of tokens
- Rare vs frequent tokens

---

**Why it matters:**
- Helps:
    - Optimize vocabulary
    - Identify noise
    - Improve embeddings

---

#### **2.3 Tokenization Impact on LLM Cost**

**Important Insight:**
LLM cost is proportional to:

**Total Tokens = Input tokens + Output tokens**

---

**Example**

"I love NLP" -> 3 tokens
"unbelievable" -> ["un", "believ", "able"] -> 3 tokens

**Same word length is not equal token count**

---

#### **Key Factors Affecting Cost**

- Tokenizer type
- Text structure
- Preprocessing decisions

---

#### **2.4 Preprocessing + Tokenization Pipeline**

**Steps**
1. Clean text (minimal)
2. Normalize (unicode, spaces)
3. Deicide what to keep/remove
4. Tokenize using appropriate tokenizer
4. Analyze token count

---

#### **Important Principle**
- For LLMs -> **minimal preprocessing**
- For ML pipelines -> **agressive preprocessing**

---

#### **2.5 Step-by-Step Workflow**

1. Input raw text
2. Apply light cleaning
3. Pass through tokenizer
4. Count tokens
5. Optimize pipeline

---

# =========================================
# 3. INTERVIEW NOTES
# =========================================

**Key Points**

- Tokenization directly impacts LLM cost and latency
- Different tokenizers produce different token counts
- Pipeline design is task-dependent

---

**Advantages**

- Optimized pipelines reduces cost 
- Better token distribution improves performance
- Enables scalable systems

---

**Disadvantages**

- Over optimization may remove context
- Tokenizer mismatch can hurt model performance

---

#### **When to Use**

- LLM applications (cost optimization)
- RAG pipelines
- Production NLP systems

---

#### **When NOT to Overdo**

- Simple experiments
- Direct API usage without scale concerns

---

#### **Important Comparisons**

**Tokenizer Comparisons**

| **Tokenizer** | **Use Case** |
|---------------|--------------|
| NLTK | Learning |
| spaCy | Production NLP |
| BPE/WordPiece | LLMs |

---

#### **Preprocessing Trade-off**

| **Approach** | **Effect** |
|--------------|------------|
| Heavy Cleaning | Less tokens, less context |
| Minimal Cleaning | More tokens, more context |

---

# =========================================
# 4. IMPLEMENTATION
# =========================================


#### **4.1 Setup**

In [9]:
import re
import nltk
import spacy
from collections import Counter
from transformers import AutoTokenizer

nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")

hf_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

[nltk_data] Downloading package punkt to /home/kartik/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


---

#### **4.2 Basic Preprocessing**

In [7]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+',' ',text)

    return text.strip()

---

#### 4.3 Tokenizer Comparison**

In [10]:
from nltk.tokenize import word_tokenize

text = "I love NLP and AI!"

# NLTK
nltk_tokens = word_tokenize(text)

# spaCy
doc = nlp(text)
spacy_tokens = [token.text for token in doc]

# HuggingFace
hf_tokens = hf_tokenizer.tokenize(text)

print("NLTK: ", nltk_tokens)
print("spaCy: ", spacy_tokens)
print("HF: ", hf_tokens)

NLTK:  ['I', 'love', 'NLP', 'and', 'AI', '!']
spaCy:  ['I', 'love', 'NLP', 'and', 'AI', '!']
HF:  ['i', 'love', 'nl', '##p', 'and', 'ai', '!']


---

#### **4.4 Token Count Comparison**

In [11]:
def count_tokens(text):
    return {
        "nltk" :len(word_tokenize(text)),
        "spacy" : len([token.text for token in nlp(text)]),
        "hf" : len(hf_tokenizer.tokenize(text))
    }

print(count_tokens(text))

{'nltk': 6, 'spacy': 6, 'hf': 7}


---

#### **4.5 Token Distribution**

In [13]:
def token_distribution(tokens):
    return Counter(tokens)

dist = token_distribution(nltk_tokens)
print(dist)

Counter({'I': 1, 'love': 1, 'NLP': 1, 'and': 1, 'AI': 1, '!': 1})


---

#### **4.6 Full Pipeline**

In [15]:
def full_pipeline(text):
    text = preprocess(text)
    tokens = hf_tokenizer.tokenize(text)
    token_count = len(tokens)

    return {
        "clean_text" :text,
        "tokens" : tokens,
        "count" : token_count
    }

print(full_pipeline("I LOVE NLP!!!"))

{'clean_text': 'i love nlp!!!', 'tokens': ['i', 'love', 'nl', '##p', '!', '!', '!'], 'count': 7}


---

#### **4.7 Variation: Cost Estimation**

In [21]:
def estimate_cost(token_count, price_per_1k = 0.002):
    return (token_count / 1000) * price_per_1k # think of 2sec and 1min/60sec

result = full_pipeline("I love NLP and AI " * 100)
print("Cost: ", estimate_cost(result["count"]))

Cost:  0.0012


---

# =========================================
# 5. GENAI / LLM / AGENT MAPPING
# =========================================


#### **In LLMs**

- Tokenization defines:
    - Input representation
    - Cost
    - Context usage

---

#### **In RAG Systems**

- Tokenization used for:
    - Chunking documents
    - Embedding generation
    - Retrieval optimization

---

#### **In AI Agents**

- Used in:
    - Prompt construction
    - Memory optimization
    - Tool invoaction

---

#### **When Not Needed**

- Small-scale tasks
- Non-LLM pipelines

---


# =========================================
# 6. MINI PROJECT / TASK
# =========================================


#### **Project: Token Cost Analyzer**

**Input** 

```
[
    "I love NLP",
    "Transformers are powerful models",
    "AI is the future"
]
```

---

**Expected Output**

- Token Counts (NLTK vs HF)
- Cost Estimation
- Token distribution

---

**Steps**
1. Preprocess text
2. Tokenize using multiple tokenizers
3. Compare token counts
4. Estimate cost
5. Analyze differences

---

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from transformers import AutoTokenizer
import re
from collections import Counter

nltk.download('punkt')

hf_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

[nltk_data] Downloading package punkt to /home/kartik/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [29]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+',' ',text)

    return text.strip()

def hugging_face_tokenizer(text):
    tokens = hf_tokenizer.tokenize(text)
    return tokens

def nltk_tokenizer(text):
    tokens = word_tokenize(text)
    return tokens

def token_counter(tokens):
    return len(tokens)

def token_distribution(tokens):
    return Counter(tokens)

def cost_estimation(token_count, price_per_1k):
    total_price = (token_count/1000) * price_per_1k

    return total_price

def full_pipeline(text, tokenizer="hf", token_price_1k = 0.05):
    text = preprocess(text)

    if tokenizer.lower()=="hf":
        tokens = hugging_face_tokenizer(text)
    
    elif tokenizer.lower() == "nltk":
        tokens = nltk_tokenizer(text)

    else:
        print("Incorrect Tokenizer")
        return None
    
    distribution = Counter(tokens)
    count = token_counter(tokens)
    price = cost_estimation(count, token_price_1k)

    return {
        "tokens" : tokens,
        "token_count" : count,
        "cost" : price,
        "token_distribution" : distribution
    }


In [35]:
input_text = [
 "I love NLP",
 "Transformers are powerful models",
 "AI is the future"
]

print("Hugging Face Tokenizer: ")
for text in input_text:
    print(text,": ",end="")
    print(full_pipeline(text))

print("-"*100,end="\n\n")

print("NLKT")

for text in input_text:
    print(text,": ",end="")
    print(full_pipeline(text, tokenizer="nltk"))


Hugging Face Tokenizer: 
I love NLP : {'tokens': ['i', 'love', 'nl', '##p'], 'token_count': 4, 'cost': 0.0002, 'token_distribution': Counter({'i': 1, 'love': 1, 'nl': 1, '##p': 1})}
Transformers are powerful models : {'tokens': ['transformers', 'are', 'powerful', 'models'], 'token_count': 4, 'cost': 0.0002, 'token_distribution': Counter({'transformers': 1, 'are': 1, 'powerful': 1, 'models': 1})}
AI is the future : {'tokens': ['ai', 'is', 'the', 'future'], 'token_count': 4, 'cost': 0.0002, 'token_distribution': Counter({'ai': 1, 'is': 1, 'the': 1, 'future': 1})}
----------------------------------------------------------------------------------------------------

NLKT
I love NLP : {'tokens': ['i', 'love', 'nlp'], 'token_count': 3, 'cost': 0.00015000000000000001, 'token_distribution': Counter({'i': 1, 'love': 1, 'nlp': 1})}
Transformers are powerful models : {'tokens': ['transformers', 'are', 'powerful', 'models'], 'token_count': 4, 'cost': 0.0002, 'token_distribution': Counter({'transfor

---

# =========================================
# 7. LEARNINGS / SUMMARY
# =========================================

#### **Key Takeaways**

- Tokenization directly impact cost and performance
- Subword tokenization is critical for LLMs
- Pipeline design must be task-aware

---

#### **Important Insight**

- More tokens = higher Cost
- Preprocessing affects tokenization
- Not all tokenizers behave the same

---

#### **Common Mistakes**

- Ignoring token count in production
- Over-cleaning text for LLMs
- Using wrong tokenizer for model
- Not analyzing token distribution

---

# =========================================
# 8. SELF-TEST QUESTIONS
# =========================================

1. **Why does tokenization impact LLM cost directly?**

Tokenization directly impacts LLM cost because pricing is based on the total number of tokens processed, which includes both input and output tokens. Different tokenizer strategies can produce different token counts for the same text, affecting how much context fits within the model's limit and how much the API charges. Efficient tokenization helps reduce unnecessary token usage while preserving the meaning, leading to lower cost and faster inference. Conversely, inefficient tokenization can inflate token count or distort structure, increasing cost and potentially degrading performance. Therefore, optimizing tokenization is critical for building cost effective and scalable LLM applications.

---

2. **How can preprocessing reduce token count but harm performance?**

Preprocessing can reduce token count by removing punctuation, stopwords or normalizing text. Which makes the input more compact. However, excessive or aggresive preprocessing can strip away important contextual and semantic information that LLMs rely on for understanding meaning. For example, removing punctuation can alter tone or intent, and remove removing certain words can break sentence structure. While fewer tokens may reduce cost, the loss of meaningful signals can lead to poorer model output. Therefore, there is a trade-off between reducing token count and preserving context, and modern LLM pipeline typically favour minimal, careful preprocessing.

---

3. **Why is it important to use same tokenizer as the model?**

It is important to use the same tokenizer as the model because the model is trained on a specific tokenization scheme, and its learned embeddings corresponds exactly to those token IDs. If a different tokenizer is used, the input text may be split into different tokens, leading to mismatched token IDs and incorrect representations. This can degrade model performance or even produce invalid outputs. Additionally, consistent tokenization ensures accurate token counting for cost estimation and proper alignment with the models vocabulary, avoiding unintended out-of-vocabulary issues or semantic distortions.